# SQRT (Sub-zero Quantitative Research Team)

## Post-Mission Data Analysis

**Author:** Caimin Keavney

The processes applied below will extract, translate and store all relevant mission data acquired by the SQRT mission. This data will be in the form of both transmitted data and data stored onboard the SD card. Depending on which of these modes is relevant, (recovery dependent) two workflows will be required to unload the data products.

### Onboard Storage (SD card)

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from array import array
import struct

In [15]:
def science_data_proc(datafile):
    """
    Extracts and cleans Science Data from MLX90640.
    """
    frames = []

    with open(datafile, "rb") as f:
        data = f.read()

    # Binary data is converted to decimal
    values = np.frombuffer(data, dtype="<f4")

    # ensure full frames only
    usable_len = (len(values) // 768) * 768
    
    values = values[:usable_len]

    frames = values.reshape(-1, 768)
    
    return frames


def house_proc(housefile):
    """
    Extracts and stores Housekeeping Data from sensors
    """
    Housekeeping_Data = {
        "Times": [],
        "Internal_Temperature": [],
        "External_Temperature": [],
        "Pressure": [],
        "GPS_Times":[],
        "Latitudes": [],
        "Longitudes": [],
        "Altitudes": [],
        "HDOP": []
    }

    with open(housefile, "r") as f:
        lines = f.readlines()

    
    for line in lines:
        items = line.strip().split(",")
        
        if len(items) == 10:
            Housekeeping_Data["Times"].append(items[0])
            Housekeeping_Data["Pressure"].append(items[2])
            Housekeeping_Data["External_Temperature"].append(items[3])
            Housekeeping_Data["Internal_Temperature"].append(items[4])
            Housekeeping_Data["GPS_Times"].append(items[5])
            Housekeeping_Data["Latitudes"].append(items[6])
            Housekeeping_Data["Longitudes"].append(items[7])
            Housekeeping_Data["Altitudes"].append(items[8])
            Housekeeping_Data["HDOP"].append(items[9])
        else:
            continue

    return Housekeeping_Data
    

    
def error_proc(error_file):
    """
    Extracts and stores Error Log
    """
    error_dict = {
        "Times": [],
        "Error": [],
        "Sensor/Class": []
    }

    with open(error_file, "r") as f:
        lines = f.readlines()[1:]

    for line in lines:
        items = line.strip().split(",")
       
             
        error_dict["Times"].append(items[0])
        error_dict["Error"].append(items[1])
        error_dict["Sensor/Class"].append(items[-1])
    
    return error_dict

def trig_proc(trig_file):
    """
    Extracts and stores Trigger file information (trigger statuses & trigger conditions if applicable).
    """

    trig_dic = {
        "time": [],
        "trig_statuses": [],
        "trig_conditions": [],
        "trig_pressures": [],
        "trig_altitudes": []
    }

    with open(trig_file, "r") as f:
        lines = f.readlines()

    for line in lines:
        items = line.strip().split(",")
        if len(items) ==5:
            trig_dic["time"].append(items[0])
            trig_dic["trig_statuses"].append(items[1])
            trig_dic["trig_conditions"].append(items[2])
            trig_dic["trig_pressures"].append(items[3])
            trig_dic["trig_altitudes"].append(items[4])
        else:
            continue
        
    return trig_dic

def time_proc(time_file):
    """
    Extracts and stores time values for thermal sensor data
    """
    times = []
    with open(time_file, "r") as f:
        lines = f.readlines()

    for line in lines:
        times.append(line)
        
    return times
    
def sd_processor(folder_path):
    """
    Processes all SD card data from inputted directory
    """
    files = os.listdir(folder_path)

    # Returns are initialised in case no files are present.
    t = d = t_s = d_s = None
    house_dict = error_dict = trig_dict = None
    
    for file in files:
        fullpath = folder_path + "/" + file
        
        if "data" in file:
            if "trig" in file:
                if "log" in file:
                    d_s = science_data_proc(fullpath)    # Science Thermal Data
                else:
                    t_s = time_proc(fullpath)    # Science Data times
            else:
                if "times_log" in file:
                    t = time_proc(fullpath)    # Normal Thermal Data times
                else:
                    d = science_data_proc(fullpath)   # Normal Thermal Data


        elif "housekeeping" in file:
            house_dict = house_proc(fullpath)     # Housekeeping data

        elif "error" in file:
            error_dict = error_proc(fullpath)   # Error Info

        elif "trigger_log" in file: 
            trig_dict = trig_proc(fullpath)    # Trigger Information

        else: 
            continue
    return house_dict, error_dict, trig_dict, (t_s,d_s), (t,d)

### Transmitted Data Packets Analysis

In [20]:
all_house_data = []
all_science_data = []


def packet_proc(textfile):
    """
    Deals with all packets 
    """
    with open(textfile, "r") as f:
        packets = f.readlines()

    for packet in packets:
        packet= packet.strip()
        
        items = packet.split("|")

        
        if items[1].strip() == "SQRT":
            packet_type = items[0].strip()[-1]
            
            if packet_type == "T":
                all_house_data.append(packet)    # Telemetry Packet
                
            elif packet_type == "D":
                all_science_data.append(packet)   # Science Packet

        else:
            continue
    house_info = house_packet_parser(all_house_data)
    frames, science_info = science_packet_parser(all_science_data)

    decoded_frames = frames_proc(frames)
    
    return house_info, science_info, decoded_frames

def house_packet_parser(packets):
    """
    Parses data from received data packets and stores/ housekeeping/science data
    """
    house_data = {
        "timestamp": [],
        "pressure": [],
        "external_temp": [],
        "internal_temp": [],
        "latitude": [],
        "longitude": [],
        "altitude": [],
        "hdop": [],
        "error_count": [],
        "error_types": []
    }
    
    for packet in packets:
        
        items = packet.split("|")
        
        if len(items)<14:
            continue
            
        house_data["timestamp"].append(items[4])
        house_data["gps_times"].append(items[5])
        house_data["latitude"].append(items[6])
        house_data["longitude"].append(items[7])
        house_data["hdop"].append(items[7])
        house_data["altitude"].append(items[8])
        house_data["internal_temp"].append(items[9])
        house_data["external_temp"].append(items[10])
        house_data["pressure"].append(items[11])
        house_data["error_count"].append(items[12])
        house_data["error_types"].append(items[13])


    return house_data

def science_packet_parser(packets):
    """
    Handles Science data packets including frame data, which is sent to save_data and saved 
    """
    frames_data = []

    trig_dic = {
        "trig": [],
        "condition": [],
        "pres": [],
        "alt": []
    }

    for packet in packets:
        items = packet.split("|")

        frames_data.append(items[2])
        trig_dic["trig"].append(items[3])
        trig_dic["condition"].append(items[4])
        trig_dic["pres"].append(items[5])
        trig_dic["alt"].append(items[6])

    return frames_data, trig_dic
    
def frames_proc(frames_list):
    """
    Processes frame data, which is in binary format
    """
    frames = []
    for frame in frames_list:
        arr = array('h')
        arr.frombytes(bytes.fromhex(frame))
        decoded_frame = [x/100 for x in arr]
        
        frames.append(decoded_frame)

        
    return frames

### Saving Frames as Figures

In [19]:
def save_frames(frames, out_dir, times=None):
    """
    frames: list of decoded frames (list of floats)
    """
    os.makedirs(out_dir, exist_ok=True) 

    fig, ax = plt.subplots()
    
    for i, frame in enumerate(frames):
        ax.clear()
    
        frame = np.array(frame)
    
        if len(frame) == 768:
            shaped = frame.reshape(24, 32)
        elif len(frame) == 48:
            shaped = frame.reshape(6, 8)
        else:
            continue
    
        heatmap = ax.imshow(shaped, cmap="inferno", origin="upper", vmin=-10, vmax = 50)
        
        label = times[i] if times is not None else f"frame_{i}"
        safe_label = str(label).replace(":", "_")
    
        ax.set_title(f"{label}")
    
        filename = os.path.join(out_dir, f"frame_{i:04d}.png")
        plt.savefig(filename, dpi=100, bbox_inches="tight")
    
    plt.close(fig)